# 🦄 VenturePulse: Global Startup Valuation & Market Intelligence
### CMSE 830 - Midterm Project Analysis & Empirical Modeling

**Research Question:** *To what extent can private tech valuations across 1,447 global unicorns (including OpenAI, Anthropic, DeepSeek, and ByteDance) be quantitatively modeled using investor network capital, sector, geographic hub, and frontier AI signals?*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("muted")

# Load processed 1,447 unicorns dataset
df = pd.read_csv("../data/processed/unicorns_with_predictions.csv")
print(f"Loaded {len(df):,} unicorns across {df['country'].nunique()} countries.")
df.head(5)

## 1. Exploratory Data Analysis: Valuation Distributions & Power Laws
We contrast the extreme skew of raw startup valuations with the linearized log-transformed distribution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["valuation_b"], bins=30, kde=True, ax=axes[0], color="#3b82f6")
axes[0].set_title("Raw Valuation ($B) - Extreme Power-Law Skew")
axes[0].set_xlabel("Valuation ($B)")

sns.histplot(np.log1p(df["valuation_m"]), bins=30, kde=True, ax=axes[1], color="#10b981")
axes[1].set_title("Log-Transformed Valuation - Approximate Normality")
axes[1].set_xlabel("log(Valuation in $M)")

plt.tight_layout()
plt.show()

## 2. Sector Valuation Breakdown
Distribution of enterprise valuations across core industry categories.

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x="clean_industry", y="valuation_b", showfliers=False, palette="viridis")
plt.title("Valuation Distribution by Sector ($B, Excluding Outliers)")
plt.xlabel("Sector")
plt.ylabel("Valuation ($B)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 3. Machine Learning Model Benchmark & Diagnostics
Reviewing the trained Random Forest and Ridge regression pipelines.

In [ ]:
with open("../models/valuation_pipeline.pkl", "rb") as f:
    bundle = pickle.load(f)

print(f"Selected Model: {bundle['model_name']}")
print(f"5-Fold CV R²: {bundle['cv_r2']:.4f}")
print(f"Test Set R²:   {bundle['test_r2']:.4f}")
print(f"Test Set MAE:  ${bundle['test_mae_m']:,.2f} Million\n")

print("Top 8 Positive Valuation Elasticities (Ridge):")
print(bundle["coefficients"].head(8).to_string(index=False))

## 4. Valuation Velocity & Decacorn Screener
Examining companies that scaled to tens or hundreds of billions in record time.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x="tenure_years",
    y="valuation_b",
    hue="efficiency_tier",
    palette={"Decacorn Alpha (Exceeds Market Trajectory)": "#10b981", "Fair Market Valuation": "#3b82f6", "Valuation Discounted / Stagnant": "#ef4444"},
    alpha=0.7
)
plt.yscale("log")
plt.title("Valuation Velocity: Operating Tenure vs Valuation ($B, Log Scale)")
plt.xlabel("Operating Horizon as Unicorn (Years)")
plt.ylabel("Valuation ($B, Log Scale)")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()